In [4]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [5]:
data = pd.read_csv("dfdata.csv")
X = data.drop("outcome", axis=1)
y = data["outcome"]

In [6]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Function to build model
def build_model(layers):
    model = Sequential()
    model.add(Dense(layers[0], input_dim=X.shape[1], activation='relu'))
    for nodes in layers[1:]:
        model.add(Dense(nodes, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Output layer
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Configurations to test
configs = [
    (1000, [4]),
    (10000, [4]),
    (100000, [4]),
    (1000, [4, 4]),
    (10000, [4, 4]),
    (100000, [4, 4]),
]

# Store results
results = []

# Main loop
for size, architecture in configs:
    print(f"Training with size {size}, architecture {architecture}")

    # Sample data
    X_train_full, X_val, y_train_full, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    X_train = X_train_full[:size % len(X_train_full)]
    y_train = y_train_full[:size % len(y_train_full)]

    model = build_model(architecture)
    start = time.time()
    history = model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0, validation_data=(X_val, y_val))
    end = time.time()

    training_error = 1 - history.history['accuracy'][-1]
    validation_error = 1 - history.history['val_accuracy'][-1]
    execution_time = end - start

    results.append({
        'Data size': size,
        'Architecture': architecture,
        'Training error': training_error,
        'Validation error': validation_error,
        'Execution time (s)': execution_time
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df)

Training with size 1000, architecture [4]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training with size 10000, architecture [4]
Training with size 100000, architecture [4]
Training with size 1000, architecture [4, 4]
Training with size 10000, architecture [4, 4]
Training with size 100000, architecture [4, 4]
   Data size Architecture  Training error  Validation error  \
0       1000          [4]         0.19100          0.186674   
1      10000          [4]         0.00360          0.003868   
2     100000          [4]         0.00158          0.001270   
3       1000       [4, 4]         0.30600          0.294971   
4      10000       [4, 4]         0.00540          0.004873   
5     100000       [4, 4]         0.00173          0.002644   

   Execution time (s)  
0          144.392631  
1          132.424411  
2          162.603471  
3          165.765235  
4          121.507137  
5          178.745989  
